# Protein Applications

> **Feature status:** Public PDB and mmCIF adapters can load an ordered
> backbone trace. A stable, general conversion from that trace to a
> domain-specific spatial graph is not yet part of the public application API.

This short notebook shows the implemented boundary with a completely offline
PDB example. It does not infer a contact network, cavity graph, domain graph,
or repulsive model from the protein. Those mappings answer different scientific
questions and must be defined and validated explicitly.


> **Notebook profile**
>
> - **Audience:** first-time users checking the biomolecular input contract.
> - **Environment:** base 0.2 development API; no optional extra is required.
> - **Network:** offline; this notebook writes a temporary local PDB file.
> - **Outputs:** text diagnostics only; no repository files are changed.
> - **Scientific boundary:** the result is an ordered backbone curve, not an automatically inferred residue-interaction graph.


## 1. What you will inspect

The example makes four checks that also apply to a real local file or an RCSB
download: atom/chain selection, source records, returned issues, and the common
embedded-graph `pos`/`pts` contract.


In [ ]:
from pathlib import Path
import tempfile

import knotted_graph
from knotted_graph.inputs import from_protein_ca_backbone

print("KnottedGraph version:", knotted_graph.__version__)


## 2. Load a local C-alpha backbone

A real PDB may contain several models, chains, atom names, and alternate
locations. This deliberately tiny file contains one model and one chain so the
selection is unambiguous. For a multi-chain structure, pass `chain_id`
explicitly; the adapter does not silently choose one for you.


In [ ]:
def pdb_atom(serial, residue, sequence, x, y, z):
    return (
        f"ATOM  {serial:5d} {'CA':>4s} {residue:>3s} A{sequence:4d}    "
        f"{x:8.3f}{y:8.3f}{z:8.3f}  1.00 20.00           C\n"
    )

temporary_directory = tempfile.TemporaryDirectory()
pdb_path = Path(temporary_directory.name) / "backbone_demo.pdb"
pdb_path.write_text(
    "".join(
        [
            pdb_atom(1, "ALA", 1, 0.0, 0.0, 0.0),
            pdb_atom(2, "GLY", 2, 1.2, 0.1, 0.0),
            pdb_atom(3, "SER", 3, 1.8, 1.0, 0.4),
            pdb_atom(4, "VAL", 4, 2.6, 1.4, 1.1),
        ]
    ),
    encoding="utf-8",
)

result = from_protein_ca_backbone(
    pdb_path,
    pdb_id="DEMO",
    chain_id="A",
    model_id=1,
)

print("selected chain:", result.chain_id)
print("available chains:", dict(result.available_chains))
print("coordinate shape:", result.coords.shape)
print("residues:", [record["residue_name"] for record in result.records])
print("issues:", result.issues)


## 3. Inspect the returned graph

The ordered samples are stored as the geometry of one open graph edge. They are
not converted into four topological graph vertices. This distinction matters:
coordinate sampling density should not change the abstract graph topology.


In [ ]:
graph = result.graph
u, v, key = next(iter(graph.edges(keys=True)))
points = graph.edges[u, v, key]["pts"]

print("graph nodes / edges:", graph.number_of_nodes(), graph.number_of_edges())
print("edge sample count:", len(points))
print("start agrees with node pos:", (points[0] == graph.nodes[u]["pos"]).all())
print("end agrees with node pos:", (points[-1] == graph.nodes[v]["pos"]).all())
print("input kind:", graph.graph["input_kind"])


## 4. Decide what your scientific graph means

If the ordered backbone is your intended curve, continue with embedding
validation and the core workflow. If you instead need a contact, residue,
cavity, or domain graph, first define: (1) what a node represents; (2) the rule
that creates an edge; (3) coordinate units and closure; and (4) perturbation or
validation checks. The library does not choose that domain mapping implicitly.

The native Repulsor layout is also a separate, optional external-backend route;
loading a PDB/mmCIF file never runs it automatically.


## 5. Continue

Read the web **Input handling** page for PDB/mmCIF atom-selection, download,
closure, and parser details. Then use **02 - Core Workflows** for graph
validation, projection, and invariant provenance. Do not close an open backbone
or interpret a zero polynomial merely to make a desired downstream calculation
possible; closure and graph construction are scientific choices.
